In [1]:
import numpy as np
import jax.numpy as jnp
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from softmobility import SoftBody, Flow, FlowBody, Field

np.set_printoptions(precision=3, linewidth=100, suppress=True, sign=" ")

## Assembly creation

### Data 

In [2]:
yaml_data = """
# Variable Prefixes (Optional)
design_names:     # Design parameter prefixes/names (extends default "design")
  - myradius      # Recognized as a parameter prefix/name (e.g., `myradius` in expressions)
  - distance
  - offset
input_names:
  - gravity       # Gravity field

# Default Values (Optional)
defaults:
  myradius0: 1.   # Parameter: Listed in `design_names`
  myradius1: 1.
  distance: -1.
  mass: 0.        # Constant: auto-detected

# Spheres (Mandatory)
spheres:
  - # Sphere 0 #################
    radius: myradius0                       
    position: [offset - myradius0 - distance/2, 0, 0]             
    force: [mass*gravity0, mass*gravity1, mass*gravity2]              

  - # Sphere 1 #################
    radius: myradius1                 
    position: [offset + myradius1 + distance/2, 0, 0]       
    force: [mass*gravity0, -mass*gravity1, -mass*gravity2]
"""

In [3]:
mybody= SoftBody(yaml_data, verbose=False)
print(repr(mybody))

SPHERE ASSEMBLY
  2 spheres
  0 degrees of freedom
  4 design parameters
  3 input parameters

Default values
  degrees of freedom dof: [] = []
  design parameters param: ['distance', 'myradius0', 'myradius1', 'offset'] = [-1.  1.  1.  0.]
  input parameters param: ['gravity0', 'gravity1', 'gravity2'] = [ 0.  0.  0.]

SPHERE 0
  radius: 1.0
  position: [-0.5  0.   0. ]
  orientation: [ 0.  0.  0.]
  force: [ 0.  0.  0.]
  torque: [ 0.  0.  0.]

SPHERE 1
  radius: 1.0
  position: [ 0.5  0.   0. ]
  orientation: [ 0.  0.  0.]
  force: [ 0.  0.  0.]
  torque: [ 0.  0.  0.]



## Fluid-structure interaction problem

In [4]:
class PureShearFlow(Flow):
    """Simple pure shear flow u = (y, 0, 0)."""

    def _velocity(self, pos):
        x, y, z = pos  # Extract coordinates
        shear_rate = self.params[0] if self.params else 1  # Default value
        return jnp.array([shear_rate * y, 0, 0])
    
class GravityField(Field):
    """Constant uniform field g = (0, 0, -param[0])."""

    def _field_vector(self, pos, time):
        gravity_acc = self.params[0] if self.params else 1  # Default value
        return jnp.array([0, 0, -gravity_acc])

In [5]:
# Create a shear flow with shear rate 1
shear_flow = PureShearFlow(1)
# Create a gravity field of acceleration 0
gravity_field = GravityField(0)

# Test it (steady case)
pos = [1.0, 2.0, 3.0]  
grad_u = shear_flow.gradient(pos)
Omega, rate_of_strain = shear_flow.omega_s(pos)  
print("Velocity Gradient Tensor (∇u):\n", grad_u)
print("Angular velocity Ω:", Omega)
print("Rate-of-strain E):\n", rate_of_strain)
print("Gravity field", gravity_field.vector(pos))

Velocity Gradient Tensor (∇u):
 [[ 0.  1.  0.]
 [ 0.  0.  0.]
 [ 0.  0.  0.]]
Angular velocity Ω: [ 0.   0.  -0.5]
Rate-of-strain E):
 [[ 0.   0.5  0. ]
 [ 0.5  0.   0. ]
 [ 0.   0.   0. ]]
Gravity field [ 0  0  0]


### Numerics

In [ ]:
# parameters
final_time = 10 
dt = 0.1

# mybody.set_design_defaults(new_dict={'offset': 0.5})

solver = FlowBody(mybody, shear_flow, gravity_field, dt=dt, init_orientation=[0, .6, 0], integrator="RK2")
trajectory = solver.simulate(T=final_time)

Time: 0.000 / 10.000  Integrator Euler


In [7]:
# Extract position, orientation and dofs
positions = jnp.array([t[0] for t in trajectory])
orientations = jnp.array([t[1] for t in trajectory])
dofs = jnp.array([t[2] for t in trajectory])

# Time vector
t = jnp.linspace(0, final_time, len(trajectory))

# Create a subplot figure with 3 rows and 1 column
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("DOF", "Position (X, Y, Z)", "Orientation (X, Y, Z)"),
    shared_xaxes=True,  # Share the x-axis for all plots (time)
    vertical_spacing=0.1  # Adjust vertical spacing (reduce clutter)
)

# Plot DOFs in the first subplot
fig.add_trace(go.Scatter(x=t, y=dofs, mode='lines', name='DOF'), row=1, col=1)

# Plot position components (X, Y, Z) in the second subplot
fig.add_trace(go.Scatter(x=t, y=positions[:, 0], mode='lines', name='Position X'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 1], mode='lines', name='Position Y'), row=2, col=1)
fig.add_trace(go.Scatter(x=t, y=positions[:, 2], mode='lines', name='Position Z'), row=2, col=1)

# Plot orientation components (X, Y, Z) in the third subplot
fig.add_trace(go.Scatter(x=t, y=orientations[:, 0], mode='lines', name='Orientation X'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 1], mode='lines', name='Orientation Y'), row=3, col=1)
fig.add_trace(go.Scatter(x=t, y=orientations[:, 2], mode='lines', name='Orientation Z'), row=3, col=1)

# Update layout for titles and labels
fig.update_layout(
    title="Trajectory Data Over Time",
    xaxis_title="Time (t)",
    # yaxis_title="Values (Position/Orientation Components)",
    template="plotly_white",  # Set the background to white
    showlegend=True,  # Show legend to distinguish between the traces
    height=800,  # Set figure height
    width=800,   # Set figure width
    plot_bgcolor='white',  # Set the plot background to white
    paper_bgcolor='white'  # Set the paper background to white
)

# Show the plot
fig.show()

### Theory

In [8]:
def Rodrigues_formula(rvec):
    """
    Rotation matrix from rotation vector r using Rodrigues' rotation formula.
    """
    rvec = np.array(rvec)
    theta = np.linalg.norm(rvec)

    if theta < 1E-6:
        return np.eye(3)

    else:
        k = rvec / theta
        kx, ky, kz = k
        K = np.array([[0, -kz, ky], [kz, 0, -kx], [-ky, kx, 0]])
        R = np.eye(3) + np.sin(theta) * K + (1 - np.cos(theta)) * np.dot(K, K)

        return R

In [13]:
phi_num = np.zeros_like(t)
theta_num = np.zeros_like(t)
p0 = np.array([1, 0, 0])

for i, ti in enumerate(t):
    R = Rodrigues_formula(orientations[i])
    p = R @ p0
    phi_num[i] = np.atan2(p[1], p[0])
    theta_num[i] = np.atan(np.sqrt((1 - p[1]**2 - p[0]**2) / (p[1]**2 + p[0]**2)))

In [14]:
tensors = mybody.compute_mobility_problem()
M0 = tensors.M @ mybody.compute_composition_of_forces()  # Mobility with forces/torques at the ref point x_0
N = mybody.Nspheres
# Reshape M into a 3D array where each element is a 6x6 matrix representing one sphere's data
M_blocks = M0.reshape(6, 6 * N).T.reshape(N, 6, 6)
M_mean = np.mean(M_blocks, axis=0)

# Bretherton parameter
Bretherton = tensors.C_E[-1,1] # * M_mean[1,1]

# print("Rigid mobility tensor\n", M_mean)
# print("Coupling with strain\n", tensors.C_E)    # Exx, Exy, Exz, Eyy, Eyz 
print("Bretherton parameters:", Bretherton)

Bretherton parameters: 0.42011255


In [15]:
time = np.linspace(0,t[-1],num=100)
phi0 = phi_num[0]
theta0 = theta_num[0]
c = np.sqrt((1 + Bretherton) / (1 - Bretherton))
K = np.sqrt(np.cos(phi0)**2 + c**2 * np.sin(phi0)) / np.tan(theta0)

phi = np.atan2(-(1/c) * np.sin(time / (c + 1/c)), np.cos(time / (c + 1/c)))
# phi = np.atan(-(1/c) * np.tan(time / (c + 1/c)))
theta = np.atan(np.sqrt(np.cos(phi)**2 + c**2 * np.sin(phi)**2) / K)

In [16]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=t, y=phi_num, mode='lines', name='Numerics phi'))
fig.add_trace(go.Scatter(x=t, y=theta_num, mode='lines', name='Numerics theta'))
fig.add_trace(go.Scatter(x=time, y=phi, mode='markers', name='Theory phi'))
fig.add_trace(go.Scatter(x=time, y=theta, mode='markers', name='Theory theta'))

fig.show()